# Notebook 22 — RLHF: un ciclo PPO minimale su TinyLlama

**Modulo:** 22 — Allineamento e generazione multimodale
**Tempo stimato:** 2-3 ore (la sezione con TinyLlama beneficia di una GPU; la sezione sul
meccanismo di PPO gira ovunque, anche su CPU)

## Cosa costruirai

Al termine di questo notebook avrai:
- Un reward euristico e completamente trasparente contro il linguaggio tossico, scritto in
  italiano (l'alternativa "leggera" descritta nella Sezione 6 della dispensa)
- Un modello TinyLlama con LoRA (Modulo 21), un modello di riferimento congelato, e un
  `PPOTrainer` configurato secondo le fasi 6-8 della dispensa
- Un ciclo di addestramento PPO reale, che genera riassunti, li valuta con il reward, e aggiorna
  la policy
- Un grafico dell'evoluzione della reward media nel tempo
- Una versione "da zero", eseguibile su CPU senza scaricare alcun modello, che implementa a mano
  la matematica del PPO (rapporto di policy, vantaggio, clipping, penalizzazione KL) su un
  problema giocattolo — utile per capire cosa succede *dentro* `PPOTrainer.step()`

## Prerequisiti

```bash
pip install transformers peft trl accelerate torch datasets python-dotenv matplotlib
```

> **Attenzione — hardware:** la sezione con TinyLlama (1,1 miliardi di parametri) e `trl` è
> pensata per una GPU (anche quella gratuita di Google Colab va bene). Su CPU funziona ma è
> lenta. Se non hai una GPU disponibile in questo momento, leggi comunque il codice — è
> corretto ed eseguibile quando la avrai — e concentrati sulla sezione finale "PPO passo-passo",
> pensata apposta per funzionare senza scaricare nulla.

Serve un file `.env` solo se vuoi autenticarti all'Hugging Face Hub (non obbligatorio per
TinyLlama, che è pubblico):
```
HUGGINGFACE_TOKEN=hf_...
```

---
## Cella 0 — Setup

Come nel Modulo 21, verifichiamo le chiavi opzionali e il device disponibile.

In [ ]:
import os
from dotenv import load_dotenv
import torch

load_dotenv()

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")  # opzionale, TinyLlama e' pubblico

def verifica_chiave(nome: str, valore: str | None) -> None:
    if valore:
        print(f"OK {nome}: {valore[:8]}...")
    else:
        print(f"NON IMPOSTATA {nome} (va bene, non e' obbligatoria per questo notebook)")

verifica_chiave("HUGGINGFACE_TOKEN", HF_TOKEN)

USA_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USA_GPU else "cpu"
print(f"\nDevice disponibile: {DEVICE}")
if not USA_GPU:
    print(
        "Nessuna GPU rilevata. La Sezione 1-4 (TinyLlama + trl) sara' molto lenta su CPU: "
        "leggi il codice comunque, ed esegui per intero la Sezione 5 ('PPO passo-passo'), "
        "pensata per funzionare senza GPU."
    )


---
## Sezione 1 — Il reward euristico anti-tossicità

Come spiegato nella dispensa (Sezione 6), un vero reward model è un classificatore addestrato
(es. una variante di RoBERTa specializzata sul linguaggio d'odio). Per tenere questo notebook
completamente autocontenuto e verificabile a mano, usiamo l'alternativa che la dispensa stessa
propone: un reward euristico basato su una lista di parole "tossiche" scelta a priori.

In [ ]:
# Lista di parole "tossiche" in italiano, scelta a priori (linguaggio offensivo lieve,
# adatta a un esercizio didattico -- non e' una lista esaustiva di moderazione)
PAROLE_TOSSICHE = [
    "idiota", "stupido", "cretino", "imbecille", "scemo",
    "stronzo", "schifezza", "deficiente", "incapace", "insulto",
]


def calcola_reward(testo: str) -> float:
    """Reward euristico: parte da 1.0 e sottrae una penalita' per ogni parola tossica
    trovata nel testo (case-insensitive). Il risultato e' clippato in [-1, 1] per evitare
    che un singolo testo molto lungo con tante ripetizioni domini l'addestramento.
    E' una fortissima semplificazione di un vero reward model appreso, ma ha il vantaggio
    di essere completamente trasparente: puoi sempre verificare a mano perche' un testo
    ha ricevuto un certo punteggio."""
    testo_minuscolo = testo.lower()
    n_occorrenze = sum(testo_minuscolo.count(parola) for parola in PAROLE_TOSSICHE)
    reward = 1.0 - 0.5 * n_occorrenze
    return max(-1.0, min(1.0, reward))


# Verifica rapida: un riassunto neutro deve valere piu' di uno che riporta insulti
testo_neutro = "Una conversazione tesa tra due colleghi su un progetto in ritardo."
testo_tossico = "Il collega ha detto che il progetto era una schifezza fatta da idioti incapaci."

reward_neutro = calcola_reward(testo_neutro)
reward_tossico = calcola_reward(testo_tossico)

print(f"Reward testo neutro:  {reward_neutro:.2f}  -> \"{testo_neutro}\"")
print(f"Reward testo tossico: {reward_tossico:.2f}  -> \"{testo_tossico}\"")
assert reward_neutro > reward_tossico, "il reward dovrebbe premiare il testo neutro"
print("\nOK: il reward euristico distingue correttamente i due casi.")


---
## Sezione 2 — Il dataset di dialoghi da riassumere

La dispensa (Sezione 6) usa DialogSum: dialoghi con un riassunto atteso, alcuni dei quali
contengono linguaggio scortese che il modello deve imparare a neutralizzare nel riassunto. Qui
ne costruiamo una versione ridotta e in italiano, sufficiente per vedere il meccanismo
all'opera.

In [ ]:
DIALOGHI = [
    {
        "dialogo": (
            "Persona A: Il tuo report e' una schifezza, sei un incapace totale.\n"
            "Persona B: Mi dispiace, lo rifaccio per domani."
        ),
    },
    {
        "dialogo": (
            "Persona A: Possiamo spostare la riunione di domani alle 15?\n"
            "Persona B: Certo, nessun problema, aggiorno il calendario."
        ),
    },
    {
        "dialogo": (
            "Persona A: Sei proprio stupido, hai sbagliato di nuovo i numeri nel foglio di calcolo.\n"
            "Persona B: Hai ragione, controllo subito e correggo l'errore."
        ),
    },
    {
        "dialogo": (
            "Persona A: Il cliente ha confermato l'ordine per la settimana prossima.\n"
            "Persona B: Ottimo, avviso subito il magazzino."
        ),
    },
    {
        "dialogo": (
            "Persona A: Sei un cretino, non hai capito niente del progetto.\n"
            "Persona B: Proviamo a rivederlo insieme con calma domani mattina."
        ),
    },
    {
        "dialogo": (
            "Persona A: Grazie per l'aiuto di ieri, mi ha salvato la giornata.\n"
            "Persona B: Figurati, e' stato un piacere."
        ),
    },
]

PROMPT_RIASSUNTO = "Riassumi la seguente conversazione in una frase, con un tono neutro:\n\n{dialogo}\n\nRiassunto:"

for esempio in DIALOGHI[:2]:
    print(PROMPT_RIASSUNTO.format(dialogo=esempio["dialogo"]))
    print("-" * 50)


---
## Sezione 3 — Caricare TinyLlama, applicare LoRA, creare il modello di riferimento

Qui replichiamo i tre modelli descritti nella dispensa (Sezione 6, tabella "I tre modelli in
gioco"). Come nel Modulo 21, applichiamo LoRA per rendere l'addestramento leggero: si aggiorna
solo una piccola frazione dei parametri, invece di tutto TinyLlama.

> Questa sezione richiede `trl`, oltre a `transformers` e `peft`. Se non l'hai installata o non
> hai una GPU a disposizione in questo momento, va comunque bene leggerla: il codice è corretto
> e la puoi eseguire in un secondo momento su Colab.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType

NOME_MODELLO = "PY007/TinyLlama-1.1B-Chat-v0.3"  # verifica su Hugging Face Hub se e' ancora disponibile

tokenizer = AutoTokenizer.from_pretrained(NOME_MODELLO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Stessa funzione di conteggio parametri introdotta nel Modulo 21
def conta_parametri(modello) -> dict:
    totali = sum(p.numel() for p in modello.parameters())
    addestrabili = sum(p.numel() for p in modello.parameters() if p.requires_grad)
    return {
        "parametri_totali": totali,
        "parametri_addestrabili": addestrabili,
        "percentuale_addestrabile": round(100 * addestrabili / totali, 4),
    }

modello_base = AutoModelForCausalLM.from_pretrained(NOME_MODELLO)

configurazione_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # proiezioni query/value dell'attenzione (Modulo 21, Sez. 3.1)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)


In [ ]:
from trl import AutoModelForCausalLMWithValueHead, create_reference_model
from peft import get_peft_model

modello_peft = get_peft_model(modello_base, configurazione_lora)
print("=== Modello principale (con LoRA) ===")
info_peft = conta_parametri(modello_peft)
for k, v in info_peft.items():
    print(f"  {k}: {v}")

# AutoModelForCausalLMWithValueHead aggiunge una piccola "testa" di valore sopra il modello LoRA:
# serve a PPO per stimare quanto e' buono uno stato durante il calcolo del vantaggio (dispensa, Sez. 6).
modello_ppo = AutoModelForCausalLMWithValueHead.from_pretrained(modello_peft)

# Il modello di riferimento (Sezione 5, Fase 8 della dispensa): una copia congelata,
# usata SOLO per calcolare la divergenza KL, mai aggiornata.
modello_riferimento = create_reference_model(modello_ppo)

print("\n=== Modello di riferimento (congelato) ===")
info_riferimento = conta_parametri(modello_riferimento)
for k, v in info_riferimento.items():
    print(f"  {k}: {v}")
print(
    "\nNota: il modello di riferimento ha 0 parametri addestrabili per costruzione -- "
    "e' una copia congelata usata solo per il confronto (divergenza KL)."
)


---
## Sezione 4 — Configurare e lanciare il ciclo PPO

`PPOConfig` fissa gli iperparametri dell'ottimizzazione (Fase 7 della dispensa); `PPOTrainer`
orchestra generazione, calcolo della reward e aggiornamento della policy in un'unica chiamata
`.step()`.

In [ ]:
from trl import PPOConfig, PPOTrainer

configurazione_ppo = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=len(DIALOGHI),
    mini_batch_size=2,
)

ppo_trainer = PPOTrainer(
    config=configurazione_ppo,
    model=modello_ppo,
    ref_model=modello_riferimento,
    tokenizer=tokenizer,
)


**Il ciclo di addestramento** (Fase 7 della dispensa, applicata passo per passo):

1. tokenizziamo i prompt del batch (i dialoghi da riassumere);
2. il modello genera un riassunto per ciascuno (`ppo_trainer.generate`);
3. calcoliamo la reward di ogni riassunto con `calcola_reward` (Sezione 1);
4. `ppo_trainer.step(...)` usa quelle reward per aggiornare la policy con PPO, penalizzando
   internamente la divergenza dal modello di riferimento (Fase 8 della dispensa).

Eseguiamo solo poche iterazioni: l'obiettivo è vedere la reward muoversi, non ottenere un
modello di produzione (la dispensa stessa, nel materiale di riferimento, usa 1.000 esempi e un
numero di epoch molto più alto).

In [ ]:
from trl.core import LengthSampler

campionatore_lunghezza = LengthSampler(15, 40)  # lunghezza variabile del riassunto generato
storico_reward = []
storico_kl = []

N_ITERAZIONI = 4  # basse di proposito: l'obiettivo e' l'intuizione, non un modello finito

for iterazione in range(N_ITERAZIONI):
    query_tensors = [
        tokenizer(PROMPT_RIASSUNTO.format(dialogo=d["dialogo"]), return_tensors="pt").input_ids[0]
        for d in DIALOGHI
    ]

    # 1-2. Generiamo un riassunto per ogni dialogo del batch
    response_tensors = []
    for query in query_tensors:
        lunghezza_target = campionatore_lunghezza()
        risposta = ppo_trainer.generate(
            query, max_new_tokens=lunghezza_target, pad_token_id=tokenizer.eos_token_id
        )
        response_tensors.append(risposta.squeeze()[len(query):])

    testi_generati = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]

    # 3. Calcoliamo la reward con la nostra funzione euristica (Sezione 1)
    reward_tensors = [torch.tensor(calcola_reward(testo)) for testo in testi_generati]

    # 4. Passo di ottimizzazione PPO vero e proprio
    statistiche = ppo_trainer.step(query_tensors, response_tensors, reward_tensors)

    reward_media = sum(r.item() for r in reward_tensors) / len(reward_tensors)
    kl_media = statistiche.get("objective/kl", float("nan"))
    storico_reward.append(reward_media)
    storico_kl.append(kl_media)

    print(f"Iterazione {iterazione+1}/{N_ITERAZIONI} | reward media: {reward_media:.3f} | KL: {kl_media:.4f}")

print("\nAndamento reward media per iterazione:", [round(r, 3) for r in storico_reward])


---
## Sezione — Visualizzare l'evoluzione della reward

In [ ]:
import matplotlib.pyplot as plt

fig, (ax_reward, ax_kl) = plt.subplots(1, 2, figsize=(11, 4))

ax_reward.plot(range(1, len(storico_reward) + 1), storico_reward, marker="o")
ax_reward.set_xlabel("Iterazione")
ax_reward.set_ylabel("Reward media")
ax_reward.set_title("Evoluzione della reward")

ax_kl.plot(range(1, len(storico_kl) + 1), storico_kl, marker="o", color="darkorange")
ax_kl.set_xlabel("Iterazione")
ax_kl.set_ylabel("Divergenza KL")
ax_kl.set_title("Deviazione dal modello di riferimento")

plt.tight_layout()
plt.show()

print(
    "Regola pratica (dispensa, Sez. 5): se la reward sale ma la KL esplode, il modello sta "
    "'sfruttando' il reward invece di migliorare davvero -- serve piu' peso alla penalizzazione KL."
)


---
## Sezione 5 — PPO passo-passo, senza modello (eseguibile ovunque, anche su CPU)

Se non hai una GPU, o vuoi semplicemente capire cosa fa `PPOTrainer.step()` "sotto il cofano",
questa sezione implementa a mano un singolo aggiornamento PPO usando solo `torch`, su un
problema giocattolo: invece di generare token uno alla volta, scegliamo direttamente tra 5
"tipi di risposta" già scritti, ciascuno con una reward calcolata con la stessa
`calcola_reward` della Sezione 1.

> **Semplificazione dichiarata:** qui trattiamo la scelta come una singola decisione (come un
> *bandit* a più braccia), non come una sequenza di token. La matematica del clipping, del
> vantaggio e della penalizzazione KL è esattamente la stessa che PPO applica token per token in
> un vero LLM — cambia solo la struttura sequenziale, che qui rimuoviamo per chiarezza.

In [ ]:
import torch

# 5 possibili "tipi di risposta" a un dialogo con linguaggio scortese, con la reward
# calcolata sullo stesso principio della Sezione 1 (qui applicata a un testo di esempio per tipo)
RISPOSTE_TIPO = [
    "riassunto neutro e conciso",
    "riassunto che riporta il linguaggio offensivo originale",
    "riassunto vago e poco informativo",
    "riassunto neutro ma inutilmente lungo e ripetitivo",
    "riassunto con insulti aggiuntivi non presenti nel dialogo originale",
]

TESTI_ESEMPIO_PER_TIPO = [
    "Una conversazione tesa su un errore nel report.",
    "Il collega ha detto che il report era una schifezza fatta da un incapace.",
    "Hanno parlato di qualcosa.",
    "Una conversazione neutra descritta con molti dettagli superflui che si ripetono piu' volte senza aggiungere informazione utile al lettore finale.",
    "Un discorso pieno di insulti, stupido cretino imbecille.",
]

reward_fissa = torch.tensor(
    [calcola_reward(t) for t in TESTI_ESEMPIO_PER_TIPO], dtype=torch.float32
)

print("Reward per ciascun tipo di risposta:")
for nome, r in zip(RISPOSTE_TIPO, reward_fissa.tolist()):
    print(f"  {r:+.2f}  {nome}")


In [ ]:
# La "policy" e' una distribuzione di probabilita' sui 5 tipi di risposta,
# rappresentata come softmax di logit addestrabili. Parte uniforme (nessuna preferenza).
logit_policy = torch.zeros(len(RISPOSTE_TIPO), requires_grad=True)

# La policy di riferimento e' fissa (equivalente al modello di riferimento congelato, Fase 8):
# qui la lasciamo uniforme per tutto l'addestramento, come "ancora" verso cui la KL viene penalizzata.
policy_riferimento = torch.softmax(torch.zeros(len(RISPOSTE_TIPO)), dim=0).detach()

ottimizzatore = torch.optim.Adam([logit_policy], lr=0.15)

EPSILON_CLIP = 0.2      # ampiezza del clipping PPO (Fase 7 della dispensa)
COEFFICIENTE_KL = 0.05  # peso della penalizzazione KL (Fase 8 della dispensa)
N_ITERAZIONI_TOY = 30

storico_reward_attesa = []
storico_kl_toy = []

for iterazione in range(N_ITERAZIONI_TOY):
    # La "policy vecchia" e' congelata per questo step di ottimizzazione (come nella dispensa:
    # PPO confronta sempre l'aggiornamento con la policy usata per generare i dati)
    policy_vecchia = torch.softmax(logit_policy.detach(), dim=0)

    policy_nuova = torch.softmax(logit_policy, dim=0)
    rapporto = policy_nuova / policy_vecchia  # "ratio" di PPO: quanto e' cambiata la probabilita' di ogni azione

    # Funzione di vantaggio (Sezione 6 della dispensa): quanto ogni risposta e' sopra/sotto la media
    vantaggio = reward_fissa - reward_fissa.mean()

    surrogato_1 = rapporto * vantaggio
    surrogato_2 = torch.clamp(rapporto, 1 - EPSILON_CLIP, 1 + EPSILON_CLIP) * vantaggio
    perdita_ppo = -torch.min(surrogato_1, surrogato_2).mean()  # obiettivo clippato (segno meno: minimizziamo)

    # Penalizzazione KL rispetto alla policy di riferimento (Fase 8: non allontanarsi troppo)
    divergenza_kl = (policy_nuova * (torch.log(policy_nuova) - torch.log(policy_riferimento))).sum()
    perdita_totale = perdita_ppo + COEFFICIENTE_KL * divergenza_kl

    ottimizzatore.zero_grad()
    perdita_totale.backward()
    ottimizzatore.step()

    reward_attesa = (torch.softmax(logit_policy.detach(), dim=0) * reward_fissa).sum().item()
    storico_reward_attesa.append(reward_attesa)
    storico_kl_toy.append(divergenza_kl.item())

    if iterazione % 5 == 0 or iterazione == N_ITERAZIONI_TOY - 1:
        print(f"Iterazione {iterazione:2d} | reward attesa: {reward_attesa:+.4f} | KL: {divergenza_kl.item():.4f}")

print(f"\nReward attesa: {storico_reward_attesa[0]:+.4f} (inizio) -> {storico_reward_attesa[-1]:+.4f} (fine)")

policy_finale = torch.softmax(logit_policy.detach(), dim=0)
print("\nProbabilita' finale per tipo di risposta:")
for nome, p in zip(RISPOSTE_TIPO, policy_finale.tolist()):
    print(f"  {p:.3f}  {nome}")


**Cosa osservare:** la reward attesa dovrebbe salire nel corso delle iterazioni, e la
probabilità dovrebbe concentrarsi sul tipo di risposta "neutro e conciso" (la reward più alta),
allontanandosi da quelli che riportano o aggiungono linguaggio offensivo — è la stessa dinamica
descritta nella Sezione 6 della dispensa ("Risultati attesi"), qui visibile in pochi secondi
invece che in ore di addestramento su GPU.

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, N_ITERAZIONI_TOY + 1), storico_reward_attesa, marker="o")
plt.xlabel("Iterazione")
plt.ylabel("Reward attesa")
plt.title("PPO 'da zero': evoluzione della reward attesa")
plt.tight_layout()
plt.show()


---
## Esercizio — completa il codice

**TODO:** aggiungi almeno 3 nuove parole alla lista `PAROLE_TOSSICHE` (Sezione 1), poi ricalcola
`calcola_reward` sui testi in `TESTI_ESEMPIO_PER_TIPO`. La reward della risposta "con insulti
aggiuntivi" dovrebbe scendere ulteriormente se una delle tue nuove parole compare in quel testo.
Verifica con un `assert`.

In [ ]:
# TODO: aggiungi almeno 3 parole alla lista qui sotto (in italiano)
NUOVE_PAROLE_TOSSICHE = []  # es. ["idiozia", "incompetente", "orrendo"]

# TODO: ricrea PAROLE_TOSSICHE includendo le nuove parole, poi ricalcola calcola_reward
# sul testo "Un discorso pieno di insulti, stupido cretino imbecille." e confronta con il
# valore ottenuto in precedenza nella Sezione 5

# reward_prima = calcola_reward("Un discorso pieno di insulti, stupido cretino imbecille.")
# PAROLE_TOSSICHE = PAROLE_TOSSICHE + NUOVE_PAROLE_TOSSICHE
# reward_dopo = calcola_reward("...")  # usa un testo che contenga una delle tue nuove parole
# assert reward_dopo <= reward_prima


---
## Sfida finale — Un mini report di addestramento riutilizzabile

Costruisci una funzione `esegui_mini_ppo_e_report` che:

1. Riceva `reward_fissa` (un tensore di reward per un insieme di "tipi di risposta") e un numero
   di iterazioni.
2. Esegua il ciclo PPO "da zero" della Sezione 5 (stessa logica: policy softmax, clipping,
   penalizzazione KL).
3. Ritorni un dizionario con: reward iniziale, reward finale, miglioramento percentuale, e la
   lista completa dello storico reward.
4. Stampi un piccolo report leggibile, sullo stile delle funzioni `calcola_costo` /
   `calcola_costo_immagine` viste nei moduli precedenti.

Completa i `# TODO`.

In [ ]:
def esegui_mini_ppo_e_report(
    reward_fissa: torch.Tensor,
    n_iterazioni: int = 30,
    epsilon_clip: float = 0.2,
    coefficiente_kl: float = 0.05,
    learning_rate: float = 0.15,
) -> dict:
    """Versione riutilizzabile del ciclo PPO 'da zero' della Sezione 5.
    Ritorna un report con reward iniziale/finale e lo storico completo."""
    n_azioni = len(reward_fissa)
    logit = torch.zeros(n_azioni, requires_grad=True)
    policy_rif = torch.softmax(torch.zeros(n_azioni), dim=0).detach()
    ottim = torch.optim.Adam([logit], lr=learning_rate)

    storico = []

    for _ in range(n_iterazioni):
        # TODO: implementa un'iterazione identica a quella della Sezione 5:
        # 1) policy_vecchia = softmax(logit.detach())
        # 2) policy_nuova = softmax(logit)
        # 3) rapporto = policy_nuova / policy_vecchia
        # 4) vantaggio = reward_fissa - reward_fissa.mean()
        # 5) surrogato_1/surrogato_2 + perdita_ppo (con torch.clamp su epsilon_clip)
        # 6) divergenza_kl rispetto a policy_rif, pesata da coefficiente_kl
        # 7) backward + step dell'ottimizzatore
        # 8) calcola la reward attesa e aggiungila a `storico`
        pass

    reward_iniziale = storico[0] if storico else float("nan")
    reward_finale = storico[-1] if storico else float("nan")
    miglioramento_percentuale = (
        100 * (reward_finale - reward_iniziale) / abs(reward_iniziale)
        if storico and reward_iniziale != 0
        else float("nan")
    )

    print(f"Reward iniziale: {reward_iniziale:+.4f}")
    print(f"Reward finale:   {reward_finale:+.4f}")
    print(f"Miglioramento:   {miglioramento_percentuale:+.1f}%")

    return {
        "reward_iniziale": reward_iniziale,
        "reward_finale": reward_finale,
        "miglioramento_percentuale": miglioramento_percentuale,
        "storico": storico,
    }


# Prova la tua implementazione (decommenta quando hai completato i TODO sopra)
# report = esegui_mini_ppo_e_report(reward_fissa, n_iterazioni=30)


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 22 — Modulo 22 del Corso Completo di IA Generativa*